# MLOps architecture on Databricks

How a model travels from raw data to a served, monitored endpoint — and how Unity Catalog and MLflow hold the whole thing together. The later notebooks implement each stage in code; this is the end-to-end reference.

## Reference architecture

Five stages, one loop. Raw data through training to serving; monitoring closes it.

```mermaid
%%{init: {"theme":"base","themeVariables":{"fontSize":"15px","primaryColor":"#ffffff","primaryBorderColor":"#D8CDBF","primaryTextColor":"#211C18","lineColor":"#C99A86","clusterBkg":"#FBF7F1","clusterBorder":"#E7DCCB"},"flowchart":{"curve":"basis","nodeSpacing":40,"rankSpacing":70,"padding":10}}}%%
flowchart LR
  RAW(["Raw data"]):::raw
  subgraph TP["Training pipeline"]
    direction LR
    FEAT["Features (feature store)"]:::train
    TRAIN["Train +<br/>MLflow"]:::train
    REG["UC registry"]:::train
    FEAT --> TRAIN --> REG
  end
  subgraph SM["Serving and monitoring"]
    direction LR
    SERVE["Serve"]:::serve
    MON["UC and Lakehouse Monitoring"]:::serve
    SERVE --> MON
  end
  RAW --> FEAT
  REG --> SERVE
  MON -->|"drift triggers retrain"| TRAIN
  UC[("Unity Catalog<br/>governance and lineage")]:::uc
  UC -.-> FEAT
  UC -.-> REG
  UC -.-> SERVE
  classDef train fill:#ffffff,stroke:#E03A15,stroke-width:1.5px,color:#211C18;
  classDef serve fill:#ffffff,stroke:#E2851F,stroke-width:1.5px,color:#211C18;
  classDef raw fill:#F3EDE5,stroke:#C9BBA8,stroke-width:1.4px,color:#4B433C;
  classDef uc fill:#FBF0DD,stroke:#E2851F,stroke-width:1.5px,color:#9A5B0C;
  linkStyle 5 stroke:#E03A15,stroke-width:1.7px;
  linkStyle 6,7,8 stroke:#E2851F,stroke-width:1.2px;
```

## Governance: MLflow and Unity Catalog

Every stage of the MLOps pipeline interfaces with two foundational systems. Maintaining a strict boundary between execution tracking and asset governance guarantees both reproducibility and compliance

| Plane | What it owns | In the loop |
|---|---|---|
| **MLflow** | Lifecycle & Tracking API: Manages experiment tracking (parameters, metrics, artifacts), model packaging (signatures, input examples), and lifecycle state | ogs every training run, packages the model, and drives promotion by shifting the @champion alias |
| **Unity Catalog** | Enterprise Governance: Provides a unified three-level namespace (catalog.schema.object) for data, features, models, and functions, enforcing strict access control and lineage. | Hosts the feature tables and registered models; acts as the absolute governance boundary dictating who can access, read, or promote assets |

The key thing: **Models in UC** means a registered model *is* a UC object (`catalog.schema.model`) — registering and governing are one act. Feature tables follow the same pattern. MLflow is the API you drive; UC is where the artifacts live.

## Deploy-code vs deploy-model

For live models that regularly need updating, you cannot just swap in a new model artifact and hope for the best. You need a controlled promotion path across environments. The first major decision is defining what actually crosses the boundary: the code that trains the model, or the trained model itself.

Two strategies:

- **Deploy-code (default)**: Promote the pipeline. The code is tested in staging, then deployed to prod where it trains and registers the model using actual production data
- **Deploy-model**: Promote the artifact. Training runs once (usually in dev); the resulting model binary is pinned and promoted through staging to prod

The following table highlights when to use each deployment method

| | Deploy-code | Deploy-model |
|---|---|---|
| What is promoted | Code (via Git/CI) | Model artifact (via MLFlow/UC) |
| Retraining cadence | Frequent or automated | Rare, expensive, or manual |
| Training-pipeline parity | Pipeline exercised in staging exactly as in prod | Training path not re-run downstream |
| Production data access | Prod data reachable for training | Prod data restricted; train where data lives |
| Regulatory reproducibility | Pipeline reproducible from versioned code | Exact artifact pinned and promoted |
| Training cost | Fine to retrain per environment | Very high; train once, reuse |


## Environments: dev / staging / prod

Talking about the promotion across environments, but what are these? 

| Environment | Purpose | Catalog | Data | Who promotes |
| :-- | :-- | :-- | :-- | :-- |
| **dev** | Fast iteration. Where code is authored, tested interactively, and pipelines are built | `dev` (or a personal sandbox) | Sampled or synthetic | Any engineer |
| **staging** | Integration and CI. Where the full pipeline is exercised exactly as it will run in production to validate PRs | `staging` | Prod-like, governed | CI on PR merge |
| **prod** | Locked-down environment where the trained model serves actual traffic | `prod` | Full production | Gated CD or an approver |


## Further reading

[`docs/links.md`](docs/links.md) holds the full annotated list. For this notebook:

- [Big Book of MLOps](https://www.databricks.com/resources/ebook/the-big-book-of-mlops): the source of the deploy-code vs deploy-model framing.
- [MLOps on Databricks (recommended architecture)](https://docs.databricks.com/machine-learning/mlops/mlops-workflow.html): the canonical reference workflow.